<a href="https://colab.research.google.com/github/raimund-buehler/visualizing-the-brain/blob/main/notebooks/day_two.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Day Two in Colab"/></a>

# Day Two: Ask Your Own Research Question

Today your group will run a small fMRI analysis for a question you choose yourself.

You will:

- choose and compare two task conditions,
- write down a prediction,
- fit a first-level model,
- calculate your own contrast,
- visualize a contrast, and
- explain what the result might mean.

The code is still guided. Your job is to make the scientific decisions and interpret the result carefully.

## You are the researchers

Yesterday, we looked at one example: button presses with the right hand compared with the left hand.

Today, each group chooses its own question. For example:

- **Seeing:** Which areas respond more strongly to visual patterns?
- **Hearing:** Which areas respond more strongly when instructions for a task are heard instead of seen?
- **Language:** Which areas respond more strongly to sentences than to simple visual patterns?
- **Calculation (challenge):** What changes in the brain when people calculate instead of understanding sentences?

Before running the analysis, your group should agree on one question and one prediction.

## Setup

Run this cell first. It installs the needed tools, loads the fMRI data, and loads the anatomical template.

In [ ]:
# @title Setup: install tools, import packages, and load data
import sys
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "nilearn", "nibabel", "matplotlib", "numpy", "pandas"
    ])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nilearn import datasets, image, plotting
from nilearn.glm.first_level import FirstLevelModel

%matplotlib inline

anat_img = datasets.load_mni152_template(resolution=2)
fmri_dataset = datasets.fetch_localizer_first_level()
fmri_img = image.load_img(fmri_dataset.epi_img)
events = pd.read_table(fmri_dataset.events)

print("Setup complete.")
print("fMRI image shape:", fmri_img.shape)
print("Number of events:", len(events))

## Choose from these task conditions

The localizer experiment uses these conditions. You can copy them exactly into the next code cell:

<img src="https://raw.githubusercontent.com/raimund-buehler/visualizing-the-brain/main/src/images/pinel_localizer_tasks.png" alt="Examples of visual and audio tasks in the Pinel localizer experiment" width="850">


```text
horizontal_checkerboard
vertical_checkerboard
sentence_reading
sentence_listening
visual_computation
audio_computation
```

Most groups should start with hearing, seeing, or language. The calculation conditions are challenge options because they can be harder to interpret.

## Choose your research question

Work in your group. Choose one question before editing the code.

Recommended starter questions:

1. **Hearing:** `sentence_listening` compared with `sentence_reading`.
2. **Seeing:** `horizontal_checkerboard` compared with `vertical_checkerboard`.
3. **Language:** `sentence_reading` compared with `horizontal_checkerboard` or `sentence_listening` compared with `horizontal_checkerboard`.
4. **Challenge:** `visual_computation` compared with `sentence_reading` or `audio_computation` compared with `sentence_listening`.

You can also swap Condition A and Condition B. That asks the opposite question, and warm and cool colors mean the opposite.

Some comparisons are easier to interpret than others. That is normal in fMRI research. A contrast can show a clear colored cluster, but the reason for that cluster can still be complicated.

The calculation conditions are challenge options. They may show areas related to numbers. But they may also show attention, effort, or task difficulty. Do not interpret them simply as "the math area".

Before you continue, discuss these four points as a group:

- Which two conditions are you comparing?
- For which condition do you expect stronger activity?
- Where in the brain do you expect the strongest activity: front/back, left/right, top/bottom?
- How did you arrive at your expectation?

## Choose Condition A and Condition B

A contrast compares two conditions:

```text
Condition A - Condition B
```

Warm colors mean: more response for **Condition A** than for **Condition B**.

Cool colors mean: more response for **Condition B** than for **Condition A**.

In the next cell, you can keep the example or replace the condition names inside the quotation marks.

In [ ]:
# Choose two conditions.
# Copy the condition names exactly and keep the quotation marks.

condition_a = "sentence_listening"
condition_b = "sentence_reading"

my_contrast = condition_a + " - " + condition_b

print("Condition A:", condition_a)
print("Condition B:", condition_b)
print("My contrast:", my_contrast)

<details>
<summary>Need ideas? Click here for examples</summary>

```python
# Hearing and language: heard sentences compared with read sentences
condition_a = "sentence_listening"
condition_b = "sentence_reading"

# Vision: one visual pattern compared with another visual pattern
condition_a = "horizontal_checkerboard"
condition_b = "vertical_checkerboard"

# Language: reading sentences compared with a simple visual pattern
condition_a = "sentence_reading"
condition_b = "horizontal_checkerboard"

# Language and hearing: hearing sentences compared with a visual pattern
condition_a = "sentence_listening"
condition_b = "horizontal_checkerboard"

# Challenge: visual calculation tasks compared with reading sentences
# This may involve numbers, but also attention and task difficulty.
condition_a = "visual_computation"
condition_b = "sentence_reading"

# Challenge: heard calculation tasks compared with hearing sentences
# This may involve numbers, but also attention and task difficulty.
condition_a = "audio_computation"
condition_b = "sentence_listening"

# Harder: hearing sentences compared with another visual pattern
# This mixes hearing/seeing, language, and visual pattern differences.
condition_a = "sentence_listening"
condition_b = "vertical_checkerboard"
```

Choose one pair. Do not run all examples at once.

</details>

## Stop and Think

Do not rush past this part. Discuss in your group:

1. What will warm colors mean for your contrast?
2. What will cool colors mean?
3. Where do you expect the strongest warm colors: front/back, left/right, top/bottom?
4. What could the colors mean: is there only one conclusion, or do the two conditions differ in other ways too?

Write down a prediction before moving on.

## Build the model

Now Nilearn checks every voxel.

This can take a short moment. You do not need to change this cell.

In [ ]:
first_level_model = FirstLevelModel(
    t_r=2.4,
    slice_time_ref=0,
    hrf_model="spm",
    drift_model="cosine",
    high_pass=0.01,
    smoothing_fwhm=4,
    minimize_memory=True,
)

first_level_model = first_level_model.fit(fmri_img, events=events)
print("Model fitted successfully.")

## Calculate your contrast

Now we calculate the contrast you selected above.

In [ ]:
my_map = first_level_model.compute_contrast(
    my_contrast,
    output_type="z_score",
)

print("Contrast calculated:", my_contrast)

## Visualize the contrast

The image below visualizes the contrast. You can click around in it interactively.

Remember:

- Warm colors mean **Condition A > Condition B**.
- Cool colors mean **Condition B > Condition A**.

Try to find the brain locations with the strongest activity!

In [ ]:
result_viewer = plotting.view_img(
    my_map,
    bg_img=anat_img,
    threshold=3.0,
    title=my_contrast,
)

result_viewer

## Give your cluster a brain-region name

The colored map shows us **where** the contrast is strong. But it does not automatically tell us the name of that brain area.

For that, we can use a **brain atlas**. A brain atlas is like a labeled map of the brain. It is not perfect, but it helps us say things like: "This peak is probably in the temporal lobe" or "This peak is near a visual area."

First, we ask Nilearn for the strongest clusters in your map.

In [ ]:
from nilearn.reporting import get_clusters_table

cluster_table = get_clusters_table(
    my_map,
    stat_threshold=3.0,
    cluster_threshold=10,
)

if len(cluster_table) == 0:
    print("No clusters found at this threshold. Try looking at the interactive map instead.")
else:
    display(cluster_table.head(8))

Now we use a simple atlas lookup.

The table below takes the peak coordinates from the cluster table and asks: **Which atlas label is closest to this point?**

Read the labels carefully. They are helpful names, not final answers.

In [ ]:
from nibabel.affines import apply_affine

atlas = datasets.fetch_atlas_harvard_oxford("cort-maxprob-thr25-2mm")
atlas_img = image.load_img(atlas.maps)
atlas_data = atlas_img.get_fdata()
atlas_labels = atlas.labels


def find_atlas_label(row):
    # Return a simple atlas label for one peak coordinate.
    xyz = np.array([row["X"], row["Y"], row["Z"]])
    voxel = np.round(apply_affine(np.linalg.inv(atlas_img.affine), xyz)).astype(int)

    if np.any(voxel < 0) or np.any(voxel >= atlas_data.shape):
        return "outside atlas"

    label_number = int(atlas_data[tuple(voxel)])
    if label_number == 0:
        return "no clear atlas label"

    return atlas_labels[label_number]


if len(cluster_table) == 0:
    print("No atlas labels because no clusters were found.")
else:
    atlas_table = cluster_table[["Cluster ID", "X", "Y", "Z", "Peak Stat"]].copy()
    atlas_table["atlas_label"] = atlas_table.apply(find_atlas_label, axis=1)
    display(atlas_table.head(8))

### Discuss

Choose one strong warm cluster and, if you have one, one strong cool cluster.

For each cluster, discuss:

- What is the atlas label?
- Is it more left, right, front, back, top, or bottom?
- Does the label fit your contrast?
- What should you be careful about before making a strong claim?

Example sentence:

> One strong cluster was near the **superior temporal gyrus**. That fits our contrast because this area is often involved in hearing and spoken language. But we should be careful because our contrast changed more than one thing.

## Interpret your result

Discuss your result as a group. Use the questions below to organize your interpretation.

### Your contrast

- Which condition was Condition A?
- Which condition was Condition B?
- What did warm colors mean for your contrast?
- What did cool colors mean for your contrast?

### Where you found activity

Use the interactive viewer to describe the strongest clusters. You do not need to be perfect. Make your best careful guess.

Quick guide:

- **Occipital lobe:** back of the brain, often important for seeing.
- **Temporal lobe:** side/lower part of the brain, often important for hearing and language.
- **Parietal lobe:** top/back part of the brain, often important for attention, space, and number-related tasks.
- **Frontal lobe:** front of the brain, often important for planning, attention, and difficult tasks.

For the strongest warm colors, discuss:

- Are they more on the left, right, or both sides?
- Are they more toward the front, middle, or back?
- Are they more lower, middle, or near the top?
- What is your best guess about the brain lobes?
- Which function might fit that lobe?

For the strongest cool colors, discuss the same questions.

## Be careful

These are real fMRI data, but our analysis is simplified.

- We used only one participant.
- Not every colored point is automatically a discovery.
- Some contrasts are easier to interpret than others.
- A contrast can compare more than one thing at the same time.

For example, `horizontal_checkerboard - vertical_checkerboard` is fairly clear: the two tasks are very similar, but the visual pattern changes. `sentence_listening - sentence_reading` changes more than one thing: hearing versus seeing, spoken language versus written language, and maybe attention. The colored clusters can still be interesting, but we need to be careful about what we claim.

The calculation contrasts are especially difficult. If you compare `visual_computation - sentence_reading`, a cluster might be related to numbers. But it might also be related to calculation being harder, needing more attention, or using memory. That is why we should say: "This area might be involved in calculation" and not: "This is the math area".

This is a common problem in fMRI research. A contrast does not explain itself. Researchers have to ask: **What exactly did we compare? What else changed between the two conditions? Could there be another explanation?**

Good science means being able to say what a result suggests and what you are still unsure about.

## Prepare your poster

Each group should be ready to explain:

1. Our research question was...
2. We compared ... minus ...
3. We predicted...
4. Warm colors meant...
5. Cool colors meant...
6. We found...
7. The atlas suggested...
8. We are still unsure about...

## Sources

- Pinel, P., Thirion, B., Meriaux, S. et al. (2007). Fast reproducible identification and large-scale databasing of individual functional cognitive networks. *BMC Neuroscience, 8*, 91. https://doi.org/10.1186/1471-2202-8-91
- [Nilearn first-level models](https://nilearn.github.io/stable/glm/first_level_model.html)
- [Nilearn plotting tools](https://nilearn.github.io/stable/modules/plotting.html)
- [Nilearn atlases](https://nilearn.github.io/stable/modules/datasets.html#atlases)